In [1]:
from google.colab import userdata
from huggingface_hub import login
import os

hf_token = userdata.get("huggingface")
login(hf_token)
os.environ["HF_TOKEN"] = hf_token

In [2]:
!pip install unsloth unsloth_zoo
!apt-get update
!apt-get install -y poppler-utils
!pip install pdf2image
!pip install decord

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,833 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [45]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [36]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # RoPE scaling internally 지원
dtype = torch.float16
load_in_4bit = True


try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit", # Qwen3 (14B) fit comfortably in a google colab 16gb vram tesla t4 gpu
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        device_map = "cuda:0", # If it still causes CUDA out of memory error, try 'cpu' or 'auto' or 'balanced'
        full_finetuning = False,
        token = hf_token
    )
except TimeoutError:
    !pip install modelscope
    import os
    os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit", # Use the original model name
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        device_map = "cuda:0", # If it still causes CUDA out of memory error, try 'cpu' or 'auto' or 'balanced'
        full_finetuning = False,
        token = hf_token
    )


==((====))==  Unsloth 2026.4.4: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [39]:
import hashlib
from PIL import Image
import numpy as np
import decord
from decord import VideoReader, cpu

def get_video_frames(video_path, num_frames=8):
  """
  colab에서 qwen 모델 로드 위해 unsloth 사용
  이때 mp4는 안정적으로 분석 어려움
  따라서 mp4는 여러 개의 이미지 프레임으로 쪼개서 모델에 input으로 제공
  """
  cache_dir = "/content/drive/MyDrive/demo/cache"
  frame_dir = os.path.join(cache_dir, "frames")
  os.makedirs(cache_dir, exist_ok=True)
  os.makedirs(frame_dir, exist_ok=True)

  video_hash = hashlib.md5(video_path.encode('utf-8')).hexdigest()

  frames_cache_file = os.path.join(cache_dir, f"{video_hash}_{num_frames}_frames.npy")
  timestamps_cache_file = os.path.join(cache_dir, f"{video_hash}_{num_frames}_timestamps.npy")

  if os.path.exists(frames_cache_file) and os.path.exists(timestamps_cache_file):
    frames = np.load(frames_cache_file)
    timestamps = np.load(timestamps_cache_file)

  else:
    vr = VideoReader(video_path, ctx=cpu(0))
    total_frames = len(vr)

    indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
    frames = vr.get_batch(indices).asnumpy()
    timestamps = np.array([vr.get_frame_timestamp(idx) for idx in indices])

    np.save(frames_cache_file, frames)
    np.save(timestamps_cache_file, timestamps)

  frame_paths = []
  for i, frame in enumerate(frames):
    frame_path = os.path.join(frame_dir, f"{video_hash}_{i:03d}.png")

    if not os.path.exists(frame_path):
      Image.fromarray(frame).save(frame_path)

    frame_paths.append(frame_path)

  return video_path, frame_paths, timestamps


In [5]:
from pdf2image import convert_from_path
import os

def pdftoimage(pdf_path):
  cache_dir = "/content/drive/MyDrive/demo/cache/pdf_images"
  os.makedirs(cache_dir, exist_ok=True)

  pdf_hash = hashlib.md5(pdf_path.encode('utf-8')).hexdigest()

  pages = convert_from_path(pdf_path, dpi=200)

  image_paths = []
  for i, page in enumerate(pages, 1):
    img_path = os.path.join(cache_dir, f"{pdf_hash}_page_{i}.png")
    page.save(img_path, "PNG")
    image_paths.append(img_path)

  return image_paths

In [49]:
from pdf2image import convert_from_path
import os

def pdftoimage(pdf_path):
  cache_dir = "/content/drive/MyDrive/demo/cache/pdf_images"
  os.makedirs(cache_dir, exist_ok=True)

  pdf_hash = hashlib.md5(pdf_path.encode('utf-8')).hexdigest()

  pages = convert_from_path(pdf_path, dpi=200)

  image_paths = []
  for i, page in enumerate(pages, 1):
    img_path = os.path.join(cache_dir, f"{pdf_hash}_page_{i}.png")
    page.save(img_path, "PNG")
    image_paths.append(img_path)

  return image_paths

def make_pdf_messages(pdf_path):
  image_paths = pdftoimage(pdf_path)
  pdf_name = os.path.basename(pdf_path)

  content = [
      {
          "type": "text",
          "text": (
              # f"다음은 {pdf_name} pdf 문서를 이미지로 변환한 것입니다. 문서의 내용을 정리해주세요. 특히, 제목, 주요 작업 절차, 안전수칙, 경고사항은 반드시 포함하세요."
              f"다음은 {pdf_name} pdf 문서를 이미지로 변환한 것입니다. 문서의 내용을 정리해주세요."
          )
      }
  ]

  for i, image_path in enumerate(image_paths, 1):
    content.append({"type": "image", "image": image_path})
    content.append({"type": "text", "text": f"PDF page {i}"})

  messages = [
      {
          "role": "user",
          "content": content
      }
  ]

  return messages

def make_mp4_messages(video_path):
  video_path, frame_paths, timestamps = get_video_frames(video_path)
  video_name = os.path.basename(video_path)

  content = [
      {
          "type": "text",
          #"text": f"다음은 {video_name} 영상에서 시간 순서대로 추출한 이미지입니다. 각 frame의 timestamp를 참고하여 영상 내용을 정리해주세요. 특히, 제목, 주요 작업 절차, 안전수칙, 경고사항은 반드시 포함하세요."
          "text": f"다음은 {video_name} 영상에서 시간 순서대로 추출한 이미지입니다. 영상 내용을 정리해주세요."
      }
  ]

  for i, (frame_path, timestamp) in enumerate(zip(frame_paths, timestamps), 1):
    if isinstance(timestamp, (list, tuple, np.ndarray)):
      timestamp = float(timestamp[0])
    else:
      timestamp = float(timestamp)

    content.append({"type": "image", "image": frame_path})
    content.append({"type": "text", "text": f"video frame {i} at timestamp {timestamp:.2f}"})

  messages = [
      {
          "role": "user",
          "content": content
      }
  ]

  return messages


In [51]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit")

def run_model(messages):
  import gc, torch
  gc.collect()
  torch.cuda.empty_cache()

  inputs = processor.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_dict = True,
    return_tensors = "pt"
  )

  inputs = inputs.to(model.device)

  generated_ids = model.generate(**inputs, max_new_tokens = 1024)
  generated_ids_trimmed = [
      out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]

  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
  print(output_text)

  return output_text[0]

저장 단위\
pdf/mp4 하나를 하나의 document(원본 자료 단위)로 두고\
각 이미지가 하나의 chunk\
검색은 chunk 단위\
왜냐하면 결과적으로 사용자에게 질문과 관련된 원본 자료를 제공해야 하니까

In [ ]:
import re

def make_documents(source_path, source_type, output):
  title = os.path.basename(source_path)




  line = {"chunk_id": ,
          "doc_id": ,
          "source_type": source_type,
          "title": title,
          "chunk_index": ,
          "content": output}

In [41]:
pdf_path = "/content/drive/MyDrive/demo/[2026-산업안전실-219]28. 지게차 사용 작업.pdf"
video_path = "/content/drive/MyDrive/demo/[2026-교육총괄실-182]1 지게차 안전벨트.mp4"

pdf_messages = make_pdf_messages(pdf_path)
pdf_outputs = run_model(pdf_messages)

import gc, torch
gc.collect()
torch.cuda.empty_cache()

mp4_messages = make_mp4_messages(video_path)
mp4_outputs = run_model(mp4_messages)

['제조업 등 사망사고 유발 SIF 고위험요인 28\n지게차 사용 작업\n\n### 제목\n제조업 등 사망사고 유발 SIF 고위험요인 28\n지게차 사용 작업\n\n### 주요 작업 절차\n1. 운행경로상의 지형 또는 구조물 (급경사, 커토, 바닥의 요철, 지반 등)\n- 작업장 내 도로를 평탄화하고 제한속도 준수\n- 안전벨트 착용\n\n']
['물론입니다. 아래는 제공된 영상의 내용을 시간 순서대로 정리한 설명입니다.\n\n---\n\n### **영상 내용 정리**\n\n**제목:** 지게차 안전벨트를 안 하면 벌어지는 일다\n\n---\n\n**1. 시작: 제목과 인물 소개 (0:00 - 0:18)**\n\n- **제목:** "지게차 안전벨트를 안 하면 벌어지는 일다"는 제목이 화면에 표시되어 있으며, 이는 영상의 주제를 명확히 하고 있습니다']


In [50]:
pdf_path = "/content/drive/MyDrive/demo/[2026-산업안전실-219]28. 지게차 사용 작업.pdf"
video_path = "/content/drive/MyDrive/demo/[2026-교육총괄실-182]1 지게차 안전벨트.mp4"

pdf_messages = make_pdf_messages(pdf_path)
pdf_outputs = run_model(pdf_messages)

import gc, torch
gc.collect()
torch.cuda.empty_cache()

mp4_messages = make_mp4_messages(video_path)
mp4_outputs = run_model(mp4_messages)

['이미지에 포함된 문서의 내용을 정리해 드리겠습니다.\n\n---\n\n### **제조업 등 사망사고 유발 SIF 고위험요인 28**\n\n#### **지게차 사용 작업**\n\n이 문서는 제조업 등에서 사망사고가 발생하는 주요 원인을 분석하고, 이를 예방하기 위한 안전 조치를 제시합니다.\n\n---\n\n#### **1. 운행경로상의 지형 또는 구조물**\n- **사고 원인**: 지게차의 운행 경로에 따라 지형 또는 구조물이 존재하여 사고가 발생할 수 있습니다.\n- **예시**: 급경사, 커브, 바닥의 요철, 지반 등.\n- **예방 조치**:\n  - 작업장 내 도로를 평탄화하고 제한속도 준수.\n  - 안전벨트 착용.\n\n---\n\n#### **2. 지게차 주용도 외로 사용**\n- **사고 원인**: 지게차를 주로 사용하지만, 다른 용도로 사용하는 경우 사고 발생 위험이 증가.\n- **예']
['물론입니다. 아래는 영상의 내용을 시간 순서대로 정리한 내용입니다.\n\n---\n\n### 영상 내용 정리\n\n**[2026-교육총괄실-182]1 지게차 안전벨트.mp4**\n\n**1. 개요 및 설정**\n- **제목**: 지게차 안전벨트를 안 하면 벌어지는 일다\n- **장소**: 교육총괄실, 실내 공간, 녹색 인조草坪\n- **주요 인물**: 두 명의 남성 (1명은 검은색 후드, 1명은 파란색 작업복)\n\n---\n\n**2. 시퀀스별 내용**\n\n**0:00**\n- **화면**: 남성 한 명이 휘어진 카운터 세트의 앞에 서 있다.\n- **내용**: "지게차 안전벨트를 안 하면 벌어지는 일다"\n- **노트**: 이 장면은 지게차 안전벨트의 중요성을 강조하는 제목과 함께 시작된다.\n\n**0:10**\n- **화면**: 두 명의 남성이 함께 서 있다']


In [ ]:
pdf_path = "/content/drive/MyDrive/demo/[2026-산업안전실-219]28. 지게차 사용 작업.pdf"
video_path = "/content/drive/MyDrive/demo/[2026-교육총괄실-182]1 지게차 안전벨트.mp4"

pdf_messages = make_pdf_messages(pdf_path)
pdf_outputs = run_model(pdf_messages)

import gc, torch
gc.collect()
torch.cuda.empty_cache()

mp4_messages = make_mp4_messages(video_path)
mp4_outputs = run_model(mp4_messages)